In [ ]:
%%configure -f
{
    "vCores": 8
}


In [ ]:
KEYS = [3870724, 3870725, 12345678, 50000004, 50000005, 50000006]
import duckdb, json, os, time
from deltalake import DeltaTable
base = "/lakehouse/default/Tables/raw_txn"
SRC = f"delta_scan('{base}')"
con = duckdb.connect(); con.execute("SET TimeZone='UTC'")
out = {}

t0 = time.time()
n, d = con.execute(f"SELECT count(*), count(DISTINCT trim(txn_id)) FROM {SRC}").fetchone()
out["rows"], out["distinct_txn_id"] = n, d
print(f"rows {n:,}  distinct txn_id {d:,}  ({time.time()-t0:.0f}s)")

t0 = time.time()
hist = dict(con.execute(f"""SELECT substr(md5(trim(txn_id)), 1, 5), count(DISTINCT trim(txn_id))
                            FROM {SRC} WHERE txn_id IS NOT NULL GROUP BY 1""").fetchall())
out["hist"] = hist
print(f"buckets {len(hist):,}  ({time.time()-t0:.0f}s)")

acts = DeltaTable(base).get_add_actions(flatten=True).to_pandas()
kmin = [c for c in acts.columns if c.startswith("min.") and c.endswith("_mirror_row_id")]
kmax = [c for c in acts.columns if c.startswith("max.") and c.endswith("_mirror_row_id")]
out["files"] = len(acts); out["keys"] = {}
for key in KEYS:
    found = []
    for _, r in acts.iterrows():
        path = os.path.join(base, r["path"])
        cnt = con.execute(f"SELECT count(*) FROM read_parquet('{path}') WHERE _mirror_row_id = {key}").fetchone()[0]
        if cnt:
            found.append({"file": r["path"][-48:], "rows_with_key": int(cnt), "num_records": int(r["num_records"]),
                          "stats_min": str(r[kmin[0]]) if kmin else None, "stats_max": str(r[kmax[0]]) if kmax else None,
                          "modified": str(r.get("modification_time"))})
    live = con.execute(f"SELECT txn_id, txn_status FROM {SRC} WHERE _mirror_row_id = {key}").fetchall()
    out["keys"][str(key)] = {"physical_files": found, "visible_rows": [[str(x) for x in v] for v in live]}
    print(key, "visible:", live, "| physical:", found)

with open("/lakehouse/default/Files/probe_mirror.json", "w") as fh:
    json.dump(out, fh)
print("written Files/probe_mirror.json")
